# §60: KIM on − off on the T2-trained host: **K5dfiT2rr** vs **K0dfiT2**, read on whole DoTA-CAP (1,129) and DAD (466)

Rule: `core/docs/final/KIM_RUN.md` **§60**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **PRELIMINARY (n = 1, s2024); printed, decides nothing; K5f stays the reference.** TAD is not read (user); DADA is not read (T2 is cut from it: in-domain).

| row | what | here |
|---|---|---|
| **K0dfiT2** | K5dfiT2's command with the four `kim.*` flags false (§53 KA1) | **train + read** (control) |
| **K5dfiT2rr** | K5dfiT2's exact command, same session (TK2: the saved run has no `env.json`) | **train + read** (KIM-on side) |
| K5dfiT2 | §37 run, saved | DAD read added here; DoTA-CAP read reused; printed only (reproduction check) |

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight (K5dfiT2 run + read, T2 inputs, DoTA-CAP and DAD bakes with T2 statistics) | no | 2-5 min |
| 4 | pytest on the code this notebook calls -- **raises if red** | no | 2-3 min |
| 5 | train K0dfiT2, then K5dfiT2rr (1,740 steps each); J10, config diffs, `mul_mil` floor, start gate | **yes** | 2 x K5dfiT2's time |
| 6 | DoTA-CAP-dev + eval (`--final`, marker `OPENED_s60.json`) for the 2 new rows; pool -> whole DoTA-CAP | **yes** | 4 x 5-10 min |
| 7 | DAD (`core.evaluate`, raw) for K0dfiT2, K5dfiT2rr, K5dfiT2 | **yes** | 3 x ~5 min |
| 8 | `zs_arm_table` (control K0dfiT2) + `dad_position_diag` + labels -> **paste `REPORTS/k5dfit2_ablation_report.md` to Claude** | no | 2-5 min |

Everything new goes to `Thesis-Final/outputs/kim_t2_fixed/`. A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: T2 data, T2 A2 bakes, KNN
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5dfiT2, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 60. KIM on − off on the T2-trained host' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), \
    'KIM_RUN.md has no §60 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'KIM_T2_OFF_ROW'), 'constants has no §60 rows -- re-upload kat-vad/'

K5       = constants.KIM_MAIN_ARM                       # "K5": names run / eval sub-folders only
SAVED    = constants.KIM_T2_FIXED_DVS_IGNORE_ROW        # K5dfiT2 (§37, saved)
ON       = constants.KIM_T2_RERUN_ROW                   # K5dfiT2rr (TK2: always the on side)
K0       = constants.KIM_T2_OFF_ROW                     # K0dfiT2
NEW_ROWS = (K0, ON)                                     # K0 first: the cheaper place to find a problem
KIM_FLAGS = constants.KIM_ABLATION_FLAGS
SEED     = constants.KIM_SEED                           # 2024
ENC      = constants.V5_ENCODER                         # V2-S
OFF      = constants.V2_OFF
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV, CAP_EVAL = constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL
CAP_ALL  = constants.DOTA_CAP_ALL_DIRNAME
STRIDE   = constants.V2_E1_STRIDE_BC                    # protocol B: s1[::3]
N_DEV, N_EVAL = constants.DOTA_CAP_DEV_CLIPS, constants.DOTA_CAP_EVAL_CLIPS
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET

OUT_F    = constants.OUTPUT_ROOT
V2_CACHE = Path(DRIVE_V2) / 'cache'
A2       = V2_CACHE / 'v2' / 'A2_S'
FINAL_A2 = Path(DRIVE_F) / 'cache' / 'v2' / 'A2_S'
# Training: K5dfiT2's (§37)
T2_DATA  = Path(DRIVE_V2) / 'data' / f'{T2NAME}_v2'
T2_KNN   = V2_CACHE / 'knn' / f'{T2NAME}_v2' / constants.KNN_CACHE_FILENAME
A2_T2    = A2 / T2NAME                                  # T2 statistics
ROOT     = OUT_F / constants.KIM_T2_FIXED_OUTPUT_DIRNAME
SAVED_RUN = ROOT / SAVED / K5 / f's{SEED}'
# DoTA-CAP inputs with T2 statistics (§37's: E3's dev bake; v2 Final's eval bake or §37's rebuild)
CAP_BAKE = {CAP_DEV: [A2 / 'DoTA_CAP_s3_from_s1'],
            CAP_EVAL: [A2 / 'DoTA_CAP_eval_s3_from_s1', FINAL_A2 / 'DoTA_CAP_eval_s3_from_s1_t2']}
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
# DAD inputs with T2 statistics (§13's bake)
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
DAD_BAKE = FINAL_A2 / f'{DAD_NAME}_s{DAD_STRIDE}_t2'
# Outputs
REPORTS  = ROOT / 'REPORTS'
MARK     = ROOT / constants.KIM_T2_ABL_OPENED_FILENAME
LOCAL    = Path('/content/k5dfit2_ablation')
LOCAL.mkdir(parents=True, exist_ok=True)


def run_dir(row):
    return ROOT / row / K5 / f's{SEED}'


def read_dir(row, split):
    '''protocol-B read of ``split`` (dev / eval / whole) for a T2 row.'''
    return ROOT / row / 'eval' / split / K5


def dad_dir(row):
    '''<row>/eval/dad/K5 -- core.evaluate writes into its s<seed> sub-folder.'''
    return ROOT / row / 'eval' / DAD_SET / K5


INPUTS = (('T2 data', T2_DATA), ('T2 KNN', T2_KNN), ('T2 A2 bake', A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5dfiT2 checkpoint', SAVED_RUN / 'checkpoint_last.pt'), ('K5dfiT2 verified', SAVED_RUN / 'verified.json'),
          ('K5dfiT2 config', SAVED_RUN / 'config.yaml'), ('K5dfiT2 metrics', SAVED_RUN / 'metrics.jsonl'),
          ('K5dfiT2 whole DoTA-CAP read', read_dir(SAVED, CAP_ALL) / 'protocol_b_readout.json'),
          ('DoTA s1', DOTA_S1), ('DoTA metadata', DOTA / 'metadata_val.json'), ('DoTA labels', DOTA / 'labels_s8'),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD bake (T2 stats, §13)', DAD_BAKE / constants.V2_INPUT_MANIFEST_FILENAME))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | new rows {NEW_ROWS} -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.data.v2_splits import load_split      # noqa: E402
from core.tools import k5_edge_report as edge   # noqa: E402
from core.tools import kim_ablation_report as ka  # noqa: E402
from core.tools.v5_stage1 import _flat          # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    '''A Drive file or dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    if ids is None:
        if not dst.exists():
            tmp = dst.with_name(dst.name + '.partial')
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(src, tmp)
            os.replace(tmp, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(d, steps):
    rec, ckpt = d / 'verified.json', d / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


# Preflight on small files only (C39)
saved_cfg = _flat(yaml.safe_load((SAVED_RUN / 'config.yaml').read_text()))
assert saved_cfg['kim.enabled'] and saved_cfg['data.dataset'] == T2NAME, 'SAVED_RUN is not K5dfiT2'
assert (saved_cfg['dvs.theta'], saved_cfg['loss.dvs_anchor_mode'], saved_cfg['kim.seam_mask']) == \
    (0.7, constants.DVS_ANCHOR_MODE_IGNORE, True), 'SAVED_RUN does not carry K5dfiT2 DVS keys'
STEPS = json.loads((SAVED_RUN / 'verified.json').read_text())['step']             # 1,740
assert STEPS == constants.NEXAR_STEP_BUDGET and saved_cfg['train.num_epochs'] == constants.KIM_T2_EPOCHS
assert is_verified(SAVED_RUN, STEPS) and metrics_rows(SAVED_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, \
    'K5dfiT2: J10'
T2_SHA = manifest(A2_T2)['train_ids_sha1']
assert (manifest(A2_T2)['crn'], manifest(A2_T2)['motion']) == (OFF, ENC), manifest(A2_T2)
CAP_IDS = {CAP_DEV: sorted(load_split(CAP_DEV)), CAP_EVAL: sorted(load_split(CAP_EVAL, final=True))}   # ids only (§60 TK6)
assert (len(CAP_IDS[CAP_DEV]), len(CAP_IDS[CAP_EVAL])) == (N_DEV, N_EVAL)
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DAD_IDS) == constants.DAD_TEST_VIDEOS, len(DAD_IDS)


def bake_ok(d, ids):
    if not (d / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        return False
    m = manifest(d)
    return ((m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, T2_SHA)
            and not [v for v in ids if not (d / f'{v}.npy').exists()])


CAP_IN = {s: next((d for d in CAP_BAKE[s] if bake_ok(d, CAP_IDS[s])), None) for s in CAP_BAKE}
assert None not in CAP_IN.values(), (f'no T2-statistics DoTA-CAP bake on all clips: {CAP_IN} -- §37 built them; '
                                     'paste this to Claude, do not rebuild')
assert bake_ok(DAD_BAKE, DAD_IDS), f'DAD bake is not T2 statistics on all {len(DAD_IDS)} videos: {manifest(DAD_BAKE)}'
miss = [v for v in CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL] if not (DOTA_S1 / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} DoTA-CAP clips have no CLIP s1 rows: {miss[:5]}'
print(f'preflight OK: K5dfiT2 step {STEPS} | T2 sha {T2_SHA[:12]} | DoTA-CAP {N_DEV} + {N_EVAL} | DAD {len(DAD_IDS)}')
print('inputs:', {s: str(d) for s, d in CAP_IN.items()}, '| DAD', DAD_BAKE)

## 4. Tests on the code this notebook calls (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim_ablation_report.py',
     'core/tests/test_k5_edge_report.py', 'core/tests/test_dota_cap_pool.py', 'core/tests/test_zs_arm_table.py',
     'core/tests/test_zs_compare.py', 'core/tests/test_kim.py', 'core/tests/test_dvs_dataset.py'])

## 5. Train **K0dfiT2**, then **K5dfiT2rr** (§60 TK1, TK2, TK4, TK5)

K0dfiT2 = K5dfiT2's `--set` list with the four `kim.*` flags false. K5dfiT2rr = the same list unchanged. Checks: J10, config diff (K5dfiT2 → K5dfiT2rr = {}; K5dfiT2rr → K0dfiT2 = the four flags), last-epoch `mul_mil` > 0.01, start gate (step-1 `mil` equal), `env.json` per row. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
# K5dfiT2's command (§37 notebook, step 6: K5T2's BASE + the four DVS keys)
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true']
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={constants.KIM_T2_EPOCHS}', 'train.amp=true',
        f'data.dataset={T2NAME}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', f'loss.dvs_anchor_mode={constants.DVS_ANCHOR_MODE_IGNORE}', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', 'dvs.theta=0.7', 'dvs.theta_ego=0.85']
SPEC = {ON: [*BASE, *K5_SETS], K0: [*BASE, *[f'{k}=false' for k in KIM_FLAGS]]}


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(row):
    d = run_dir(row)
    rows = metrics_rows(d / 'metrics.jsonl')
    assert ckpt_step(d / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    if row == ON:
        diff = ka.config_diff(saved_cfg, got)
        assert not diff, f'{row}: differs from {SAVED} by {diff} (TK5 allows nothing)'
        print(f'{row}: config == {SAVED}')
    else:
        on_cfg = _flat(yaml.safe_load((run_dir(ON) / 'config.yaml').read_text()))   # trained above
        print(f'{row}: KIM flags off vs {ON}:', ka.check_kim_only(on_cfg, got, row))
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    ref = metrics_rows(SAVED_RUN / 'metrics.jsonl')
    for key in ('mil', 'mul_mil', 'kim_mil'):
        print(f'  last-epoch {key} {last_epoch(rows, key):.4f} ({SAVED} {last_epoch(ref, key):.4f})')


def train_row(row):
    d = run_dir(row)
    if is_verified(d, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    print(f'{row}: trains on {env}')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SPEC[row] for x in ('--set', s)],
         '--data-dir', stage(T2_DATA, LOCAL / 'stage' / 't2_data'), '--clip-dir', stage(A2_T2, LOCAL / 'stage' / 't2_a2'),
         '--knn-cache', stage(T2_KNN, LOCAL / 'stage' / 'knn.npz'), '--output-dir', local, *resume],
        log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / name, d / name)
    flush_drive()
    assert ckpt_step(d / 'checkpoint_last.pt') == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': (d / 'checkpoint_last.pt').stat().st_size}))
    publish(local / 'verified.json', d / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')


for row in NEW_ROWS:
    train_row(row)
check_run(ON)                       # config == K5dfiT2 first, so K0's diff is read against the on row
check_run(K0)

st = edge.start_check(run_dir(ON), run_dir(K0))
START = 'matched' if st['same_draws'] else 'UNMATCHED START'
saved_mil = metrics_rows(SAVED_RUN / 'metrics.jsonl')[0]['mil']
ENVS = {r: (json.loads((run_dir(r) / constants.KIM_RUN_ENV_FILENAME).read_text())
            if (run_dir(r) / constants.KIM_RUN_ENV_FILENAME).exists() else 'not recorded') for r in (*NEW_ROWS, SAVED)}
TK4 = {'status': START, 'step1_on': st['control'], 'step1_off': st['new'], 'step1_saved': saved_mil, 'env': ENVS}
print(f"TK4 step-1 mil: {ON} {st['control']:.6f} | {K0} {st['new']:.6f} -> {START} | {SAVED} (saved) {saved_mil:.6f}")
print('env:', ENVS)

## 6. DoTA-CAP reads: dev + eval (`--final`, §60 TK6) for K0dfiT2 / K5dfiT2rr, then pool each -> whole DoTA-CAP (1,129)

In [ ]:
from core.tools import dota_cap_pool  # noqa: E402

if not MARK.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    MARK.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%MZ', time.gmtime()), 'split': CAP_EVAL,
                                'rows': list(NEW_ROWS), 'seed': SEED, 'authorized': 'user, 2026-11-28',
                                'rule': 'KIM_RUN.md §60: printed, decides nothing',
                                'checkpoints': {r: str(run_dir(r) / 'checkpoint_last.pt') for r in NEW_ROWS}}, indent=2))
L_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL])
L_IN = {s: stage(CAP_IN[s], LOCAL / 'stage' / f'cap_{s}', CAP_IDS[s]) for s in CAP_IN}
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')

for row in NEW_ROWS:
    ckpt = run_dir(row) / 'checkpoint_last.pt'
    for split in (CAP_DEV, CAP_EVAL):
        out = read_dir(row, split)
        if not (out / 'protocol_b_readout.json').exists():
            local = LOCAL / 'eval' / row / split
            shutil.rmtree(local, ignore_errors=True)
            final = ['--final'] if split == CAP_EVAL else []
            run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt, *final,
                 '--input-dir', L_IN[split], '--s1-dir', L_S1, '--metadata', DOTA / 'metadata_val.json',
                 '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', split, '--stride', STRIDE,
                 '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_{split}.log')
            publish(local, out)
        r = json.loads((out / 'protocol_b_readout.json').read_text())
        assert r['split'] == split and r['clips'] == len(CAP_IDS[split]) and r['global_steps'] == {f's{SEED}': STEPS}, r
        assert (r['arm']['crn'], r['arm']['motion']) == (OFF, ENC), r['arm']
    out = read_dir(row, CAP_ALL)
    if not (out / 'protocol_b_readout.json').exists():
        local = LOCAL / 'eval' / row / CAP_ALL
        shutil.rmtree(local, ignore_errors=True)
        dota_cap_pool.write_pooled(read_dir(row, CAP_DEV), read_dir(row, CAP_EVAL), local, (N_DEV, N_EVAL))
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    print(f"{row}: whole DoTA-CAP pooled ({r['clips']} clips) | macro {r['macro']['mean']:.4f}")
flush_drive()

## 7. DAD (466 test videos, s7, raw; the §13 bake with T2 statistics) for K0dfiT2, K5dfiT2rr and the saved K5dfiT2

In [ ]:
L_DAD_META = stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta')
L_DAD_BAKE = stage(DAD_BAKE, LOCAL / 'stage' / 'dad_bake', DAD_IDS)
DAD_ROWS = (*NEW_ROWS, SAVED)
for row in DAD_ROWS:
    out = dad_dir(row) / f's{SEED}'
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / DAD_SET
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', run_dir(row) / 'checkpoint_last.pt',
             '--set', f'data.dataset={DAD_NAME}', '--data-dir', L_DAD_META, '--clip-dir', L_DAD_BAKE,
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_dad.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(DAD_IDS), r
    print(f"{row} DAD: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
flush_drive()

## 8. `zs_arm_table` (control **K0dfiT2**) on whole DoTA-CAP + DAD, `dad_position_diag`, §53 KA5 labels -> **paste `REPORTS/k5dfit2_ablation_report.md` to Claude**

In [ ]:
from core.tools import zs_arm_table as zat  # noqa: E402

local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
NAME = constants.KIM_T2_ABL_TABLE_NAME
ROWS = (K0, ON, SAVED)
SETS = (zat.DOTA, zat.DAD)
SRC = {zat.DOTA: lambda row: read_dir(row, CAP_ALL), zat.DAD: dad_dir}
old_pool, old_name = zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA]
zat.POOLING[zat.DOTA] = f'{N_DEV + N_EVAL} clips, protocol B; micro / AP after per-clip min-max'
zat.SET_NAME[zat.DOTA] = 'DoTA-CAP (dev+eval)'
try:
    reads = [a for row in ROWS for s in SETS for a in ('--read', s, row, str(SRC[s](row)))]
    zat.main([*reads, '--control', K0, '--seed', str(SEED), '--name', NAME, '--out-dir', str(local)])
finally:
    zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = old_pool, old_name
table_md = local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md'
table_md.write_text(table_md.read_text().replace('DoTA-CAP-dev and DADA-2000 after', 'DoTA-CAP (dev+eval) after'))
tab = json.loads((local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.json').read_text())['sets']

diag = local / constants.KIM_T2_ABL_DAD_DIAG_DIRNAME
run([sys.executable, '-m', 'core.tools.dad_position_diag', *[x for r in ROWS for x in ('--row', r, dad_dir(r) / f's{SEED}')],
     '--control', K0, '--seed', SEED, '--out-dir', diag])


def label(c):
    # §53 KA5
    if c['mean'] >= constants.V5_MIN_GAIN and c['low'] > 0:
        return constants.KIM_DVS_IGNORE_GAIN
    if c['mean'] <= -constants.V5_MIN_GAIN and c['high'] < 0:
        return constants.KIM_DVS_IGNORE_COST
    return constants.KIM_DVS_IGNORE_NO_DIFFERENCE


def cell(c, labelled=True):
    tag = label(c) if labelled else 'printed'
    return f"{c['mean']:+.4f} [{c['low']:+.4f}, {c['high']:+.4f}] {tag}"


SET_NAME = {zat.DOTA: 'DoTA-CAP (dev+eval, 1,129)', zat.DAD: 'DAD (466, raw; macro = position, read beside the placebo)'}
labelled = START == 'matched'
lines = ['# §60 KIM on − off on the T2-trained host (KIM_RUN.md §60), s2024, n = 1 — printed, decides nothing; K5f stays', '',
         f"Corpus T2 ({T2NAME}), {STEPS} steps per row. KIM-on side **{ON}**; start gate (TK4): **{START}** "
         f"(step-1 mil on {TK4['step1_on']:.6f}, off {TK4['step1_off']:.6f}; saved {SAVED} {TK4['step1_saved']:.6f}).", '',
         '| row | what | env |', '|---|---|---|']
what = {K0: 'VideoMAE V2-S, KIM off (control)', ON: 'K5dfiT2 re-run, KIM on', SAVED: 'K5dfiT2 saved (§37), printed'}
for row in ROWS:
    lines.append(f'| {row} | {what[row]} | {ENVS[row]} |')
lines += ['', '## Cells (macro [95 %] / micro / AP)', '', '| row | ' + ' | '.join(SET_NAME[s] for s in SETS) + ' |',
          '|---|---|---|']
for row in ROWS:
    cs = []
    for s in SETS:
        c = tab[s]['cells'][row]
        cs.append(f"{c['macro']['mean']:.4f} [{c['macro']['low']:.4f}, {c['macro']['high']:.4f}] / {c['micro']:.4f} / {c['ap']:.4f}")
    lines.append(f'| {row} | ' + ' | '.join(cs) + ' |')
lines += ['', '## KIM = K5dfiT2rr − K0dfiT2 (paired; §53 KA5: GAIN ≥ +0.01 and interval above 0, COST mirror, else NO DIFFERENCE)', '',
          '| contrast | set | Δ micro | Δ macro | Δ AP (point) |', '|---|---|---|---|---|']
for arm, text, lab in ((ON, f'{ON} − {K0} (KIM)', labelled), (SAVED, f'{SAVED} (saved) − {K0}, printed', False)):
    for s in SETS:
        d = tab[s]['deltas'][arm]
        ap = tab[s]['cells'][arm]['ap'] - tab[s]['cells'][K0]['ap']
        mac = cell(d['macro'], lab and s == zat.DOTA)   # DAD macro only beside the placebo (TK7)
        lines.append(f"| {text} | {SET_NAME[s]} | {cell(d['micro'], lab)} | {mac} | {ap:+.4f} |")
if not labelled:
    lines += ['', '**UNMATCHED START: no contrast is labelled (TK4).**']
rep = tab[zat.DOTA]['cells'][ON]['macro']['mean'] - tab[zat.DOTA]['cells'][SAVED]['macro']['mean']
lines += ['', f'Reproduction (TK2, printed): {ON} − {SAVED} whole DoTA-CAP macro {rep:+.4f} (expectation |Δ| ≤ 0.005).',
          'Nexar-host reference (other corpus, printed, not a contrast): §53.1 K5dfi KIM on − off micro DoTA −0.0041 n.s., '
          'DAD −0.0036 n.s.; §58.1 DoTA host: NO DIFFERENCE on DoTA-CAP-dev and DADA.', '',
          '## DAD position diagnostics (control K0dfiT2; §56 DA5)', '', (diag / 'dad_position_diag.md').read_text(), '',
          table_md.read_text(), '']
REPORT = local / f'{constants.KIM_T2_ABL_REPORT_STEM}.md'
REPORT.write_text('\n'.join(lines) + '\n')
(local / 'tk4.json').write_text(json.dumps(TK4, indent=1))
publish(local, REPORTS / 'k5dfit2_ablation')
publish(REPORT, REPORTS / REPORT.name)
print((REPORTS / REPORT.name).read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{REPORT.name} to Claude')